# Notebook 02 — Motion Recognition Training
## Detectra AI · FYP Project

**Platform:** Kaggle (P100 GPU, 16 GB VRAM)  
**Model:** `MCG-NJU/videomae-base` fine-tuned on UCF-101  
**Output:** `motion_videomae_quantized.onnx` (INT8)  

### Pipeline
```
UCF-101 dataset
    └─ extract 16-frame clips @ 8 fps, 224×224
        └─ VideoMAE fine-tune (20 epochs, AdamW + cosine LR)
            └─ ONNX export  (1, 16, 3, 224, 224)
                └─ INT8 quantisation  →  motion_videomae_quantized.onnx
```

---
> **Runtime checklist before running**  
> 1. Kaggle → Settings → Accelerator → **GPU P100**  
> 2. Add dataset: *UCF101* (search in **Add Data**)  
> 3. Enable internet access (required for HuggingFace download)

## 1 · Environment Setup

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

pip('transformers>=4.40.0', 'accelerate>=0.29.0')
pip('torch==2.2.2', 'torchvision==0.17.2', '--index-url', 'https://download.pytorch.org/whl/cu118')
pip('onnx>=1.16.0', 'onnxruntime-gpu>=1.17.0')
pip('av>=12.0.0')          # PyAV — zero-copy video decoding
pip('scikit-learn', 'tqdm', 'matplotlib', 'seaborn')

print('All packages installed.')

In [ ]:
import os, re, random, warnings, json, time
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import av
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

from transformers import (
    VideoMAEForVideoClassification,
    VideoMAEImageProcessor,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Configuration

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
UCF_ROOT      = Path('/kaggle/input/UCF101/UCF-101')   # adjust if dataset name differs
SPLITS_DIR    = Path('/kaggle/input/UCF101/ucfTrainTestlist')
OUTPUT_DIR    = Path('/kaggle/working')
CKPT_PATH     = OUTPUT_DIR / 'motion_videomae_best.pt'
ONNX_FP32     = OUTPUT_DIR / 'motion_videomae_fp32.onnx'
ONNX_INT8     = OUTPUT_DIR / 'motion_videomae_quantized.onnx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── model / training hyper-params ──────────────────────────────────────────
MODEL_NAME    = 'MCG-NJU/videomae-base'
NUM_FRAMES    = 16          # frames per clip
FRAME_RATE    = 8           # fps to sample from video
IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_EPOCHS    = 20
LR            = 1e-4
WEIGHT_DECAY  = 1e-2
WARMUP_RATIO  = 0.1
SEED          = 42

# ── sliding-window evaluation ──────────────────────────────────────────────
WINDOW_STRIDE = 8           # frames overlap between consecutive windows

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Configuration loaded.')

## 3 · UCF-101 Class Definitions

In [ ]:
UCF101_CLASSES = [
    'ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam',
    'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress',
    'Biking', 'Billiards', 'BlowDryHair', 'BlowingCandles', 'BodyWeightSquats',
    'Bowling', 'BoxingPunchingBag', 'BoxingSpeedBag', 'BreastStroke', 'BrushingTeeth',
    'CleanAndJerk', 'CliffDiving', 'CricketBowling', 'CricketShot', 'CuttingInKitchen',
    'Diving', 'Drumming', 'Fencing', 'FieldHockeyPenalty', 'FloorGymnastics',
    'FrisbeeCatch', 'FrontCrawl', 'GolfSwing', 'Haircut', 'HammerThrow',
    'Hammering', 'HandstandPushups', 'HandstandWalking', 'HeadMassage', 'HighJump',
    'HorseRace', 'HorseRiding', 'HulaHoop', 'IceDancing', 'JavelinThrow',
    'JugglingBalls', 'JumpRope', 'JumpingJack', 'Kayaking', 'Knitting',
    'LongJump', 'Lunges', 'MilitaryParade', 'Mixing', 'MoppingFloor',
    'Nunchucks', 'ParallelBars', 'PizzaTossing', 'PlayingCello', 'PlayingDaf',
    'PlayingDhol', 'PlayingFlute', 'PlayingGuitar', 'PlayingPiano', 'PlayingSitar',
    'PlayingTabla', 'PlayingViolin', 'PoleVault', 'PommelHorse', 'PullUps',
    'Punch', 'PushUps', 'Rafting', 'RockClimbingIndoor', 'RopeClimbing',
    'Rowing', 'SalsaSpin', 'ShavingBeard', 'Shotput', 'SkateBoarding',
    'Skiing', 'Skijet', 'SkyDiving', 'SoccerJuggling', 'SoccerPenalty',
    'StillRings', 'SumoWrestling', 'Surfing', 'Swing', 'TableTennisShot',
    'TaiChi', 'TennisSwing', 'ThrowDiscus', 'TrampolineJumping', 'Typing',
    'UnevenBars', 'VolleyballSpiking', 'WalkingWithDog', 'WallPushups',
    'WritingOnBoard', 'YoYo',
]

assert len(UCF101_CLASSES) == 101, f'Expected 101 classes, got {len(UCF101_CLASSES)}'

CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(UCF101_CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

print(f'Total classes : {len(UCF101_CLASSES)}')
print(f'First 5       : {UCF101_CLASSES[:5]}')

## 4 · Dataset — Video Reader & UCF-101 Loader

In [ ]:
# ── ImageNet normalisation constants ───────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def read_video_av(
    video_path: str,
    num_frames: int = NUM_FRAMES,
    fps: int = FRAME_RATE,
) -> Optional[torch.Tensor]:
    """
    Decode `num_frames` frames uniformly sampled from the video.

    Returns
    -------
    Tensor of shape (T, C, H, W) with values in [0, 1], or None on error.
    """
    try:
        container = av.open(video_path)
        stream    = container.streams.video[0]
        total_frames = stream.frames

        # fall back to duration-based estimate when frame count is unavailable
        if total_frames == 0:
            duration_s   = float(stream.duration * stream.time_base)
            total_frames = max(1, int(duration_s * stream.average_rate))

        # uniformly spaced indices across the video
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
        index_set = set(indices.tolist())

        frames, frame_idx = [], 0
        for packet in container.demux(stream):
            for frame in packet.decode():
                if frame_idx in index_set:
                    img = frame.to_ndarray(format='rgb24')       # (H, W, 3) uint8
                    frames.append(img)
                frame_idx += 1

        container.close()

        if len(frames) == 0:
            return None

        # pad / trim to exactly num_frames
        while len(frames) < num_frames:
            frames.append(frames[-1])
        frames = frames[:num_frames]

        # (T, H, W, 3) → (T, 3, H, W) float32 in [0,1]
        tensor = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).float() / 255.0
        return tensor

    except Exception as e:
        return None


def build_transforms(is_train: bool) -> transforms.Compose:
    """Per-frame spatial augmentation + normalisation."""
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True)]
    if is_train:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.RandomErasing(p=0.25),
        ]
    ops.append(transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))
    return transforms.Compose(ops)


print('Video reader utilities defined.')

In [ ]:
class UCF101Dataset(Dataset):
    """
    UCF-101 dataset wrapper.

    Discovers videos by walking UCF_ROOT/<ClassName>/*.avi,
    then optionally filters to an explicit split file from ucfTrainTestlist.
    """

    def __init__(
        self,
        root: Path,
        split: str = 'train',          # 'train' | 'test'
        split_idx: int = 1,            # 1, 2, or 3
        splits_dir: Optional[Path] = None,
        transform: Optional[transforms.Compose] = None,
        num_frames: int = NUM_FRAMES,
    ):
        self.root       = root
        self.transform  = transform
        self.num_frames = num_frames

        # ── Build file list ────────────────────────────────────────────────
        if splits_dir and (splits_dir / f'{split}list0{split_idx}.txt').exists():
            split_file = splits_dir / f'{split}list0{split_idx}.txt'
            self.samples = self._load_from_split_file(split_file, root)
        else:
            # fall back: walk directory
            self.samples = self._walk_directory(root, split)

        print(f'[UCF101Dataset] {split:>5} split  →  {len(self.samples):,} clips')

    # ── helpers ────────────────────────────────────────────────────────────

    def _load_from_split_file(
        self, split_file: Path, root: Path
    ) -> List[Tuple[Path, int]]:
        samples = []
        with open(split_file) as fh:
            for line in fh:
                parts = line.strip().split()
                if not parts:
                    continue
                rel_path = parts[0]                        # e.g. Archery/v_Archery_g01_c01.avi
                class_name = rel_path.split('/')[0]
                if class_name not in CLASS_TO_IDX:
                    continue
                label   = CLASS_TO_IDX[class_name]
                vid_path = root / rel_path
                if vid_path.exists():
                    samples.append((vid_path, label))
        return samples

    def _walk_directory(
        self, root: Path, split: str
    ) -> List[Tuple[Path, int]]:
        """
        Walk root/<ClassName>/*.avi and do an 80/20 train/test split per class.
        Used when official split files are not available.
        """
        all_videos: Dict[str, List[Path]] = {}
        for cls in UCF101_CLASSES:
            cls_dir = root / cls
            if cls_dir.is_dir():
                vids = sorted(cls_dir.glob('*.avi'))
                if vids:
                    all_videos[cls] = vids

        samples = []
        rng = random.Random(SEED)
        for cls, vids in all_videos.items():
            n_train = max(1, int(0.8 * len(vids)))
            rng.shuffle(vids)
            chosen = vids[:n_train] if split == 'train' else vids[n_train:]
            for v in chosen:
                samples.append((v, CLASS_TO_IDX[cls]))
        return samples

    # ── Dataset interface ─────────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        vid_path, label = self.samples[idx]
        frames = read_video_av(str(vid_path), num_frames=self.num_frames)

        if frames is None:
            # return a black clip on read error so we never crash a batch
            frames = torch.zeros(self.num_frames, 3, IMG_SIZE, IMG_SIZE)
        else:
            # apply per-frame spatial transform
            if self.transform:
                frames = torch.stack([self.transform(f) for f in frames])
            else:
                frames = torch.stack([
                    transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True)(f)
                    for f in frames
                ])

        return frames, label   # (T, C, H, W), int


print('UCF101Dataset class defined.')

In [ ]:
# ── Instantiate datasets ───────────────────────────────────────────────────
train_transform = build_transforms(is_train=True)
val_transform   = build_transforms(is_train=False)

train_dataset = UCF101Dataset(
    root=UCF_ROOT, split='train', split_idx=1,
    splits_dir=SPLITS_DIR, transform=train_transform,
)
val_dataset = UCF101Dataset(
    root=UCF_ROOT, split='test', split_idx=1,
    splits_dir=SPLITS_DIR, transform=val_transform,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val   batches : {len(val_loader):,}')

In [ ]:
# ── Quick sanity check: visualise one clip ─────────────────────────────────
frames_sample, label_sample = train_dataset[0]

print(f'Clip tensor shape : {frames_sample.shape}  (T x C x H x W)')
print(f'Label             : {label_sample}  ({IDX_TO_CLASS[label_sample]})')
print(f'Value range       : [{frames_sample.min():.3f}, {frames_sample.max():.3f}]')

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for t, ax in enumerate(axes.flat):
    frame = (frames_sample[t] * std + mean).clamp(0, 1)
    ax.imshow(frame.permute(1, 2, 0).numpy())
    ax.axis('off')
    ax.set_title(f't={t}', fontsize=7)

fig.suptitle(f'Class: {IDX_TO_CLASS[label_sample]}', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_clip.png', dpi=100)
plt.show()

## 5 · Model — VideoMAE Fine-Tuning

In [ ]:
# ── Load pre-trained VideoMAE ──────────────────────────────────────────────
processor = VideoMAEImageProcessor.from_pretrained(MODEL_NAME)

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_NAME,
    num_labels=101,
    id2label=IDX_TO_CLASS,
    label2id=CLASS_TO_IDX,
    ignore_mismatched_sizes=True,   # classifier head is replaced
)
model = model.to(DEVICE)

# Parameter counts
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total/1e6:.1f} M')
print(f'Trainable params : {trainable/1e6:.1f} M')

In [ ]:
# ── Freeze backbone for the first 5 epochs (gradual unfreeze) ─────────────
def set_backbone_requires_grad(model, requires_grad: bool):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = requires_grad

set_backbone_requires_grad(model, requires_grad=False)

# ── Optimiser & scheduler ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR * 10,       # higher LR while backbone is frozen
    weight_decay=WEIGHT_DECAY,
)

total_steps   = NUM_EPOCHS * len(train_loader)
warmup_steps  = int(WARMUP_RATIO * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Total training steps : {total_steps:,}')
print(f'Warmup steps         : {warmup_steps:,}')

## 6 · Training Loop

In [ ]:
UNFREEZE_EPOCH = 5    # epoch at which we unfreeze the backbone


def train_one_epoch(model, loader, optimizer, scheduler, criterion, epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [train]', leave=False)

    for clips, labels in pbar:
        # clips: (B, T, C, H, W)  — VideoMAE expects (B, T, C, H, W)
        clips  = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(pixel_values=clips)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * clips.size(0)
        preds  = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += clips.size(0)

        pbar.set_postfix(loss=f'{running_loss/total:.4f}', acc=f'{correct/total:.3f}')

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='Validation', leave=False)

    for clips, labels in pbar:
        clips  = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        outputs = model(pixel_values=clips)
        loss    = criterion(outputs.logits, labels)

        running_loss += loss.item() * clips.size(0)
        preds  = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += clips.size(0)

    return running_loss / total, correct / total


print('Training functions defined.')

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Gradual unfreeze at epoch UNFREEZE_EPOCH ───────────────────────────
    if epoch == UNFREEZE_EPOCH + 1:
        print(f'\n[Epoch {epoch}] Unfreezing backbone with LR = {LR:.1e}')
        set_backbone_requires_grad(model, requires_grad=True)
        for pg in optimizer.param_groups:
            pg['lr'] = LR

    t0 = time.time()
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, epoch
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # ── Save best checkpoint ───────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {'epoch': epoch, 'state_dict': model.state_dict(),
             'optimizer': optimizer.state_dict(), 'val_acc': val_acc},
            CKPT_PATH,
        )
        marker = '  ← best'
    else:
        marker = ''

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
        f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
        f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}  '
        f'({elapsed:.0f}s){marker}'
    )

# save training history
with open(OUTPUT_DIR / 'training_history.json', 'w') as fh:
    json.dump(history, fh, indent=2)

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Learning curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train', marker='o', ms=4)
ax1.plot(epochs_range, history['val_loss'],   label='Val',   marker='o', ms=4)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax1.axvline(UNFREEZE_EPOCH, color='red', linestyle='--', alpha=0.4, label='unfreeze')

ax2.plot(epochs_range, history['train_acc'], label='Train', marker='o', ms=4)
ax2.plot(epochs_range, history['val_acc'],   label='Val',   marker='o', ms=4)
ax2.set_title('Top-1 Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
ax2.axvline(UNFREEZE_EPOCH, color='red', linestyle='--', alpha=0.4)

plt.suptitle('VideoMAE Fine-Tuning on UCF-101', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=120)
plt.show()

## 7 · Sliding-Window Evaluation on Test Set

In [ ]:
def extract_all_clips(
    video_path: str,
    num_frames: int = NUM_FRAMES,
    stride: int = WINDOW_STRIDE,
) -> List[torch.Tensor]:
    """
    Extract overlapping clips using a sliding window.
    Returns a list of tensors, each of shape (T, C, H, W).
    """
    try:
        container = av.open(video_path)
        stream = container.streams.video[0]
        raw_frames = []
        for frame in container.decode(stream):
            raw_frames.append(frame.to_ndarray(format='rgb24'))
        container.close()
    except Exception:
        return []

    if len(raw_frames) < num_frames:
        while len(raw_frames) < num_frames:
            raw_frames.append(raw_frames[-1])
        return [_frames_to_tensor(raw_frames[:num_frames])]

    clips = []
    for start in range(0, len(raw_frames) - num_frames + 1, stride):
        clip_frames = raw_frames[start: start + num_frames]
        clips.append(_frames_to_tensor(clip_frames))
    return clips


def _frames_to_tensor(frames: List[np.ndarray]) -> torch.Tensor:
    t = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).float() / 255.0
    return val_transform(t.unsqueeze(0)).squeeze(0) if False else \
           torch.stack([val_transform(f) for f in t])


@torch.no_grad()
def sliding_window_predict(model, video_path: str) -> int:
    """Majority-vote over all window clips."""
    clips = extract_all_clips(video_path)
    if not clips:
        return 0

    model.eval()
    all_logits = []
    for clip in clips:
        clip = clip.unsqueeze(0).to(DEVICE)   # (1, T, C, H, W)
        logits = model(pixel_values=clip).logits
        all_logits.append(logits.cpu())

    avg_logits = torch.stack(all_logits).mean(dim=0)
    return avg_logits.argmax(dim=-1).item()


print('Sliding-window evaluation functions defined.')

In [ ]:
# ── Reload best checkpoint ─────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}  (val_acc={ckpt["val_acc"]:.4f})')

# ── Run sliding-window inference on a random subset (max 500 videos) ───────
MAX_EVAL_VIDEOS = 500
eval_samples = val_dataset.samples[:MAX_EVAL_VIDEOS]

all_preds, all_labels = [], []
for vid_path, label in tqdm(eval_samples, desc='Sliding-window eval'):
    pred = sliding_window_predict(model, str(vid_path))
    all_preds.append(pred)
    all_labels.append(label)

sw_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'\nSliding-window Top-1 accuracy : {sw_acc:.4f} ({int(sw_acc*len(all_labels))}/{len(all_labels)})')

In [ ]:
# ── Per-class classification report (top 20 classes by frequency) ─────────
from collections import Counter

label_counts = Counter(all_labels)
top20_labels = [l for l, _ in label_counts.most_common(20)]
top20_names  = [IDX_TO_CLASS[l] for l in top20_labels]

mask = [i for i, l in enumerate(all_labels) if l in top20_labels]
sub_preds  = [all_preds[i]  for i in mask]
sub_labels = [all_labels[i] for i in mask]

print(classification_report(
    sub_labels, sub_preds,
    labels=top20_labels,
    target_names=top20_names,
    zero_division=0,
))

In [ ]:
# ── Confusion matrix (top-20 classes) ────────────────────────────────────
cm = confusion_matrix(sub_labels, sub_preds, labels=top20_labels)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=top20_names, yticklabels=top20_names, ax=ax,
)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Top-20 UCF-101 Classes')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120)
plt.show()

## 8 · ONNX Export

In [ ]:
# ── Wrapper to expose plain tensor → tensor interface ─────────────────────
class VideoMAEONNXWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.model(pixel_values=pixel_values).logits


model.eval()
wrapper = VideoMAEONNXWrapper(model).to(DEVICE)

dummy_input = torch.randn(1, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

torch.onnx.export(
    wrapper,
    (dummy_input,),
    str(ONNX_FP32),
    input_names=['pixel_values'],
    output_names=['logits'],
    dynamic_axes={
        'pixel_values': {0: 'batch_size'},
        'logits':       {0: 'batch_size'},
    },
    opset_version=17,
    do_constant_folding=True,
)

# ── Verify the exported model ──────────────────────────────────────────────
onnx_model = onnx.load(str(ONNX_FP32))
onnx.checker.check_model(onnx_model)
print(f'FP32 ONNX model saved and verified  →  {ONNX_FP32}')
print(f'Model size: {ONNX_FP32.stat().st_size / 1e6:.1f} MB')

In [ ]:
# ── Validate FP32 ONNX output matches PyTorch output ─────────────────────
sess = ort.InferenceSession(str(ONNX_FP32), providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

dummy_np = dummy_input.cpu().numpy()
ort_out  = sess.run(None, {'pixel_values': dummy_np})[0]

with torch.no_grad():
    pt_out = wrapper(dummy_input).cpu().numpy()

max_diff = np.abs(ort_out - pt_out).max()
print(f'Max absolute difference PyTorch vs ONNX : {max_diff:.6f}  (threshold 1e-3)')
assert max_diff < 1e-3, 'ONNX output deviates too much from PyTorch!'

## 9 · INT8 Quantisation

In [ ]:
# ── Dynamic INT8 quantisation via onnxruntime ─────────────────────────────
# Dynamic quantisation does not require a calibration dataset;
# weights are quantised statically, activations at inference time.

quantize_dynamic(
    model_input=str(ONNX_FP32),
    model_output=str(ONNX_INT8),
    weight_type=QuantType.QInt8,
    optimize_model=True,
)

fp32_size = ONNX_FP32.stat().st_size / 1e6
int8_size = ONNX_INT8.stat().st_size / 1e6
print(f'FP32 model : {fp32_size:.1f} MB')
print(f'INT8 model : {int8_size:.1f} MB  ({100*int8_size/fp32_size:.1f}% of FP32)')

In [ ]:
# ── Latency + accuracy comparison FP32 vs INT8 ────────────────────────────
N_RUNS = 20

sess_fp32 = ort.InferenceSession(str(ONNX_FP32), providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession(str(ONNX_INT8), providers=['CPUExecutionProvider'])

inp = dummy_np  # shape (1, 16, 3, 224, 224)

# warm-up
for _ in range(3):
    sess_fp32.run(None, {'pixel_values': inp})
    sess_int8.run(None, {'pixel_values': inp})

t0 = time.time()
for _ in range(N_RUNS):
    out_fp32 = sess_fp32.run(None, {'pixel_values': inp})[0]
lat_fp32 = (time.time() - t0) / N_RUNS * 1000

t0 = time.time()
for _ in range(N_RUNS):
    out_int8 = sess_int8.run(None, {'pixel_values': inp})[0]
lat_int8 = (time.time() - t0) / N_RUNS * 1000

drift = np.abs(out_fp32 - out_int8).max()
print(f'FP32 latency : {lat_fp32:.1f} ms/clip')
print(f'INT8 latency : {lat_int8:.1f} ms/clip  (speedup {lat_fp32/lat_int8:.2f}x)')
print(f'Max logit drift FP32→INT8 : {drift:.4f}')

In [ ]:
# ── INT8 accuracy on a small validation slice ─────────────────────────────
N_EVAL = 200
int8_correct = 0

for vid_path, label in tqdm(val_dataset.samples[:N_EVAL], desc='INT8 accuracy eval'):
    frames, _ = val_dataset[val_dataset.samples.index((vid_path, label))]
    x = frames.unsqueeze(0).numpy()     # (1, T, C, H, W)
    logits = sess_int8.run(None, {'pixel_values': x})[0]
    pred = logits.argmax(axis=-1).item()
    if pred == label:
        int8_correct += 1

int8_acc = int8_correct / N_EVAL
print(f'INT8 Top-1 accuracy ({N_EVAL} clips): {int8_acc:.4f}')
print(f'PT  Top-1 accuracy (full val)       : {best_val_acc:.4f}')

## 10 · Save Artefacts & Summary

In [ ]:
# ── Save class index mapping ───────────────────────────────────────────────
mapping = {'idx2class': IDX_TO_CLASS, 'class2idx': CLASS_TO_IDX}
with open(OUTPUT_DIR / 'ucf101_class_mapping.json', 'w') as fh:
    json.dump(mapping, fh, indent=2)

# ── Save model config ─────────────────────────────────────────────────────
config = {
    'model_name'    : MODEL_NAME,
    'num_classes'   : 101,
    'num_frames'    : NUM_FRAMES,
    'frame_rate'    : FRAME_RATE,
    'img_size'      : IMG_SIZE,
    'imagenet_mean' : IMAGENET_MEAN,
    'imagenet_std'  : IMAGENET_STD,
    'best_val_acc'  : best_val_acc,
    'onnx_fp32'     : str(ONNX_FP32),
    'onnx_int8'     : str(ONNX_INT8),
}
with open(OUTPUT_DIR / 'motion_model_config.json', 'w') as fh:
    json.dump(config, fh, indent=2)

# ── Final summary ─────────────────────────────────────────────────────────
print('=' * 60)
print('TRAINING COMPLETE — ARTEFACTS SAVED')
print('=' * 60)
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    unit = 'MB' if size > 1e6 else 'KB'
    val  = size / 1e6 if size > 1e6 else size / 1e3
    print(f'  {f.name:<45}  {val:7.1f} {unit}')

## 11 · Inference Helper (for downstream integration)

In [ ]:
class MotionRecognitionInference:
    """
    Lightweight inference wrapper around the exported INT8 ONNX model.
    Drop this class into the Detectra backend.

    Usage
    -----
    recognizer = MotionRecognitionInference('motion_videomae_quantized.onnx')
    result = recognizer.predict_video('path/to/video.mp4')
    # {'label': 'Diving', 'confidence': 0.87, 'one_hot': [0, 0, ..., 1, ..., 0]}
    """

    def __init__(self, onnx_path: str, top_k: int = 5):
        self.session = ort.InferenceSession(
            onnx_path,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
        )
        self.top_k     = top_k
        self.transform = build_transforms(is_train=False)

    def preprocess(self, video_path: str) -> Optional[np.ndarray]:
        frames = read_video_av(video_path)
        if frames is None:
            return None
        processed = torch.stack([self.transform(f) for f in frames])
        return processed.unsqueeze(0).numpy()  # (1, T, C, H, W)

    def predict_video(self, video_path: str) -> Dict:
        inp = self.preprocess(video_path)
        if inp is None:
            return {'error': 'Could not decode video'}

        logits    = self.session.run(None, {'pixel_values': inp})[0][0]  # (101,)
        probs     = np.exp(logits) / np.exp(logits).sum()                 # softmax
        top_idx   = np.argsort(probs)[::-1][:self.top_k]

        # one-hot vector for the top-1 class
        one_hot        = np.zeros(101, dtype=np.float32)
        one_hot[top_idx[0]] = 1.0

        return {
            'label'      : IDX_TO_CLASS[top_idx[0]],
            'confidence' : float(probs[top_idx[0]]),
            'top_k'      : [{IDX_TO_CLASS[i]: float(probs[i])} for i in top_idx],
            'one_hot'    : one_hot.tolist(),   # 101-d vector → FusionEngine input
        }


# ── Quick demo ────────────────────────────────────────────────────────────
if val_dataset.samples:
    demo_path, demo_label = val_dataset.samples[0]
    recognizer = MotionRecognitionInference(str(ONNX_INT8))
    result = recognizer.predict_video(str(demo_path))
    print('Demo prediction:')
    print(f'  Ground truth : {IDX_TO_CLASS[demo_label]}')
    print(f'  Prediction   : {result.get("label")}  (conf={result.get("confidence", 0):.3f})')
    print(f'  Top-5        : {result.get("top_k")}')

---

## Summary

| Artefact | Description |
|---|---|
| `motion_videomae_best.pt` | Best PyTorch checkpoint (20 epochs) |
| `motion_videomae_fp32.onnx` | Full-precision ONNX export |
| `motion_videomae_quantized.onnx` | **INT8 quantised ONNX** — used by Detectra backend |
| `ucf101_class_mapping.json` | `idx↔class` lookup table |
| `motion_model_config.json` | Preprocessing & model metadata |
| `training_history.json` | Per-epoch loss/accuracy |

**Next step →** Copy `motion_videomae_quantized.onnx` to the Detectra backend at  
`detectra-ai/backend/models/motion_videomae_quantized.onnx`  
and run **Notebook 03** to train the Fusion Engine.